# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset elements (record sets, fields, columns) are referenced by their unique `@id` identifiers per the Croissant schema best practices.

### Dataset Source

The dataset is described in Croissant format and accessible at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

> **Note:** This notebook demonstrates recommended `mlcroissant` usage and basic EDA patterns. Please install the required packages if running this notebook.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and discover its structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print("Description:")
print(metadata.description)

# View publication date
if hasattr(metadata, 'datePublished'):
    print(f"\nPublished: {metadata.datePublished}")

# View fields and attributes
print("\nDataset Attributes (some):")
for attr in ['keywords', 'license', 'author', 'version']:
    value = getattr(metadata, attr, None)
    if value:
        print(f"- {attr}: {value}")

## 2. Data Overview

**Inspect record sets and discover fields/columns.**

Below, we enumerate all record sets and their associated field `@id`s. All entity references use their Croissant `@id` for clarity and reusability in code. 

In [ ]:
# List all available record sets, their fields, and field @ids
print("Record Sets in this dataset:")
record_sets = list(dataset.metadata.recordSet or [])
if not record_sets:
    print("No top-level record sets listed directly; trying distributions...")
    # Try discovering recordSets through distributions (common Croissant pattern)
    for dist in getattr(dataset.metadata, 'distribution', []):
        if hasattr(dist, 'recordSet'):
            recsets = dist.recordSet
            for rs in recsets:
                print(f"- @id: {rs['@id']}  (name: {rs.get('name', '-')})")
                # If fields available, enumerate by @id
                if 'field' in rs:
                    print("  Fields and their @id:")
                    for f in rs['field']:
                        print(f"    - {f.get('@id', '?')}  (name: {f.get('name', 'unknown')})")
        else:
            # Might be direct CSVs with columns in distribution/fileObject
            print(f"Distribution: {getattr(dist, '@id', dist)}")
            if hasattr(dist, 'fileObject') and hasattr(dist.fileObject, 'column'):
                cols = dist.fileObject.column
                print(f"  Columns/fields:")
                for col in cols:
                    print(f"    - {col['@id']}  (name: {col.get('name', 'unknown')})")
else:
    for rs in record_sets:
        print(f"- @id: {rs['@id']}  (name: {rs.get('name', '-')})")
        fields = rs.get('field', [])
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    - @id: {field['@id']}  (name: {field.get('name', '-')})")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
# For the FAIR^2 dataset, we discovered that the record set is present in the CSV (distribution -> fileObject) as columns.

# List columns for each distribution
if hasattr(dataset.metadata, "distribution"):
    for i, dist in enumerate(dataset.metadata.distribution):
        print(f"Distribution {i+1}: @id = {getattr(dist, '@id', None)}")
        fo = getattr(dist, 'fileObject', None)
        if fo and hasattr(fo, 'column'):
            print("  Columns (fields by @id):")
            for col in fo.column:
                col_id, col_name = col['@id'], col.get('name', '?')
                print(f"    - {col_id}  (name: {col_name})")

## 3. Data Extraction

Here we extract data from the main record set. For this dataset, the primary data is in a single table found in the only (first) `distribution`. Each column's unique `@id` is available above for reference.

**We'll load the table as a DataFrame and display column names.**

In [ ]:
# Identify the main record set by schema rules.
main_record_set_id = None
dist0 = None
if hasattr(dataset.metadata, 'distribution'):
    dist0 = dataset.metadata.distribution[0]
    if hasattr(dist0, 'fileObject') and hasattr(dist0.fileObject, 'recordSet'):
        main_record_set_id = dist0.fileObject.recordSet[0]['@id']
    elif hasattr(dist0, 'recordSet'):
        main_record_set_id = dist0.recordSet[0]['@id']

# If still not found, fallback: try known id (from schema)
if not main_record_set_id:
    # Deeply nested, or available directly via dataset.records()
    main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'  # This is the @id for the main distribution

print(f"Main record set @id: {main_record_set_id}\n")
# Extract all data from this record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print("Columns (fields as @id):")
for c in df.columns:
    print(f"- {c}")
print("\nPreview of data:")
df.head()

## 4. Exploratory Data Analysis (EDA)

We'll conduct a few basic operations using the DataFrame. First, select a numeric field by its column `@id`, then filter, normalize, and group.

**Example: We will use the protein coverage percentage (`coverage_percent`) if available, else another numeric column. Please check the column `@id` in the output above, adjust as needed.**

In [ ]:
# Identify numeric fields present in the columns
import numpy as np
candidate_numeric_ids = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, (int, float, np.integer, np.floating))).any() or pd.to_numeric(df[c], errors='coerce').notnull().sum() > 0]
if candidate_numeric_ids:
    numeric_field_id = candidate_numeric_ids[0]
    print(f"Selected numeric field for EDA: {numeric_field_id}")
else:
    raise ValueError('No numeric fields found in the main dataframe. Please check column ids and update selection accordingly.')

# Convert field to float for analysis if needed
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Set threshold (median as example)
threshold = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (median):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} (z-score normalization):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a label/string field if exists
# Suggest 'description', 'accession', or similar
group_field_id = None
for possible_label in ['description', 'accession', 'majority_protein_ids', 'sample_id']:
    for c in df.columns:
        if possible_label in c:
            group_field_id = c
            break
    if group_field_id:
        break

if group_field_id:
    print(f"\nGrouping by: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head()
    print("Mean of numeric field by group:")
    print(grouped_df)
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization

We visualize the distribution of the selected numeric field and the top groupings if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

if group_field_id:
    # Top N group means
    top_group = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(x=top_group.index.astype(str), y=top_group.values)
    plt.xticks(rotation=90)
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.title(f'Top 10 {group_field_id} by mean {numeric_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded, explored, and summarized key elements of the FAIR<sup>2</sup> mass spectrometry dataset using Croissant and `mlcroissant`.
- All fields and record sets were referenced by `@id`, following schema conventions.
- You can adapt this notebook template to process any Croissant dataset: just refer to field, record set, or column IDs as demonstrated above.

**For more advanced usage, see further Croissant schema documentation or explore additional EDA and ML methods!**